# **GENETIC ALGORITHM (GA) with Elitism**

For this implementation we will mimic natural evolution for the algorithm to do:

* Create a random population of solutions
* Evaluate how good each solution is
* Select the best ones to reproduce
* Combine and mutate them (very little)
* Repeat until finding a good solution

For the solutions, we will need to versions of the GA:

* Binary encoding (0s and 1s)
* Real encoding (decimals)

And test them on 3 functions:

* Test Problem 1
* Rastrigin (n=5)
* Ackley (n=2)




---



## 1. Imports

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

MASTER_SEED=42 # make results more reproducible

## 2. Objective functions and its parameters

In [26]:
def test_problem1(x):
  x1, x2 = x[0], x[1]
  return 100 * (x1**2 - x2)**2 + (1 - x1)**2

def rastrigin(x,A=10):
  n=len(x)
  return A*n+np.sum(x**2-A*np.cos(2*np.pi*x))

def ackley(x,a=20,b=1/5,c=2*np.pi):
  n=len(x)
  sum_sq=np.sum(x**2)
  sum_cos=np.sum(np.cos(c*x))
  return -a*np.exp(-b*np.sqrt(sum_sq/n))-np.exp(sum_cos/n)+a+np.e


PROBLEMS={
    'TestProblem1':{
        'func':test_problem1,
        'n_vars':2,
        'bounds':[(-2.048,2.048)]*2,
        'Optimum':0.0
    },
    'Rastrigin_n5':{
        'func':rastrigin,
        'n_vars':5,
        'bounds':[(-5.12,5.12)]*5,
        'optimum':0.0
    },
    'Ackley_n2':{
        'func':ackley,
        'n_vars':2,
        'bounds':[(-32.768,32.768)]*2,
        'optimum':0.0
    }
}

## 3. Binary encoding GA

In [18]:

# HOW MANY BITS ARE NEEDED PER VARIABLE
def compute_bits_per_var(bounds,precision=6):
  bits_list=[]
  for(lo,hi) in bounds:
    n_bits=int(np.ceil(np.log2((hi-lo)*10**precision))) # distinct values
    bits_list.append(n_bits)
  return bits_list # list of ints


# CONVERT BINARY CHROMOSOME INTO A REAL-VALUED SOLUTION VECTOR
def decode_binary(chromosome, bounds, bits_list):
  x=np.zeros(len(bounds))
  start=0
  for i, (lo,hi) in enumerate(bounds):
    n_bits=bits_list[i]
    segment=chromosome[start:start+n_bits]
    k=segment.dot(2**np.arange(n_bits-1,-1,-1))
    x[i]=lo+k*(hi-lo)/(2**n_bits-1)
    start+=n_bits
  return


# CREATE RAND POPULATION OF BINARY CHROMO
def init_binary_population(pop_size,total_bits):
  return np.random.randint(0,2,size=(pop_size,total_bits))


# DCODE VERY INDIVIDUAL AND COMPUTE ITS FITNESS
def evaluate_binary_population(pop,func,bounds,bits_list):
  fitness=np.zeros(len(pop))
  for i,chrom in enumerate(pop):
    x=decode_binary(chrom,bounds,bits_list)
    fitness[i]=func(x)
  return


# SELECTION
def roulette_wheel_selection(fitness):
  shifted=np.max(fitness)-fitness+1e-10 # for mini
  probs=shifted/np.sum(shifted) # probabilities
  return np.random.choice(len(fitness),p=probs) # smapled index


# CROSSOVER
def single_point_crossover(parent1,parent2,pc=0.9):
  if np.random.rand()<pc:
    cut=np.randome.randint(1,len(parent1)) # cut point
    child1=np.concatenate([parent1[:cut],parent2[cut:]])
    child2=np.concatenate([parent2[:cut],parent1[cut:]])
  else:
    child1,child2=parent1.copy(),parent2.copy() # no cross
  return child1,child2


# MUTATION
def binary_mutation(chromosome,pm):
  mutated=chromosome.copy()
  for j in range(len(mutated)):
    if np.random.rand()<pm:
      mutated[j]=1-mutated[j]
    return mutated

## 4. Real encoding GA

In [12]:

# RANDOM POPULATION IN REAL-NUM SPACE
def init_real_population(pop_size,bounds):
  n_vars=len(bounds)
  pop=np.zeros((pop_size,n_vars))
  for j,(lo,hi) in enumerate(bounds):
    pop[:,j]=np.random.uniform(lo,hi,pop_size)
  return pop


# COMPUTE FITNESS FOR EVERY INDIVIDUAL
def evaluate_real_population(pop,func):
  return np.array([func(ind) for ind in pop])


# SELECTION BINARY TOURNAMENT
def binary_tournament_selection(fitness):
  i,j=np.random.choice(len(fitness),size=2,replace=False)
  return i if fitness[i]<=fitness[j] else j


# CROSSOVER SBX FOR REAL-ENCODED INDIVIDUALS
def sbx_crossover(parent1,parent2,bounds,pc=0.9,nc=20):
  if np.random.rand()<pc:
    return parent1.copy(),parent2.copy()

  child1=np.zeros_like(parent1)
  child2=np.zeros_like(parent2)

  for i,(lo,hi) in enumerate(bounds):
    u=np.random.rand()
    if u<=0.5:
      beta=(2*u)**(1.0/(nc+1))
    else:
      beta=(1.0/(2*(1-u)))**(1.0/(nc+1))

    p1,p2=parent1[i],parent2[i]
    child1[i]=0.5*((1+beta)*p1+(1-beta)*p2)
    child2[i]=0.5*((1-beta)*p1+(1+beta)*p2)

    child1[i]=np.clip(child1[i],lo,hi)
    child2[i]=np.clip(child2[i],lo,hi)

  return child1,child2


# MUTATION POLYNOMIAL MUTATION
def polynomial_mutation(individual,bounds,pm,nm=20):
  mutated=individual.copy()
  for i, (lo,hi) in enumerate(bounds):
    if np.random.rand()<pm:
      u=np.random.rand()
      if u<0.5:
        delta=(2*u)**(1.0/(nm+1))-1
      else:
        delta=1-(2*(1-u))**(1.0/(nm+1))
      mutated[i]=np.clip(mutated[i]+delta*(hi-lo),lo,hi)
  return mutated

## 5. Automatic stopping criterion

In [13]:

# stops the GA when there's no meaningful imrpovement
class StagnationStopper:

  def __init__(self,patience=50,tol=1e-8):
    self.patience=patience # how many stagnant gens to tolerate
    self.tol=tol # min improvement
    self.best_so_far=np.inf # track best fitness
    self.stagnant_gen=0 # consecutive stagn gens

  def should_stop(self,current_best):
    if self.best_so_far-current_best>self.tol:
      self.best_so_far=current_best
      self.stagnant_gen=0
    else:
      self.stagnant_gen+=1

    return self.stagnant_gen>=self.patience

  def reset(self):
    self.best_so_far=np.inf
    self.stagnant_gen=0

## 6. MAIN GA loop

for both binary and real GAs

**Binary GA**

In [19]:

# binary encoded GA
def run_binary_ga(func,
                  bounds,
                  n_gens,
                  pop_size=100,
                  pc=0.9,
                  precision=6,
                  use_elitism=True,
                  verbose=False):

  # setup
  bits_list=compute_bits_per_var(bounds,precision)
  total_bits=sum(bits_list)
  pm=1.0/total_bits
  stopper=StagnationStopper(patience=50,tol=1e-8)

  # initialization
  pop=init_binary_population(pop_size,total_bits)
  fitness=evaluate_binary_population(pop,func,bounds,bits_list)
  history=[np.min(fitness)]

  # generation loop
  for gen in range(1,n_gens+1):

    # elitism: save best individual before placement
    if use_elitism:
      elite_idx=np.argmin(fitness)
      elite_chrom=pop[elite_idx].copy()
      elite_fit=fitness[elite_idx]

    # build new pop
    new_pop=[]
    while len(new_pop)<pop_size:
      idx1=roulette_wheel_selection(fitness)
      idx2=roulette_wheel_selection(fitness)
      p1,p2=pop[idx1],pop[idx2]

      # cross
      c1,c2=single_point_crossover(p1,p2,pc=pc)

      # mut
      c1=binary_mutation(c1,pm)
      c2=binary_mutation(c2,pm)

      new_pop.append(c1)
      if len(new_pop)<pop_size:
        new_pop.append(c2)

    pop=np.array(new_pop)
    fitness=evaluate_binary_population(pop,func,bounds,bits_list)

    # eli, replace worst with elite
    if use_elitism:
      worst_idx=np.argamax(fitness)
      if fitness[worst_idx]>elite_fit: # only if elite is better
        pop[worst_idx]=elite_chrom
        fitness[worst_idx]=elite_fit


    # record best fitness this gen
    best_gen=np.min(fitness)
    history.append(best_gen)

    if verbose:
      print(f'gen{gen:4d}|best fitness: {best_gen:.8f}')


    # stopping criterion check
    if stopper.should_stop(best_gen):
      if verbose:
        print(f'stopping at gen {gen} (stagn detected)')
      break

  best_idx=np.argmin(fitness)
  best_x=decode_binary(pop[best_idx],bounds,bits_list)
  best_f=fitness[best_idx]
  return best_x,best_f,history,gen

After building the new generation, we replace the **worst** individual with the **best** from the last generation. This guarantees the best solution is never lost.

Real encoding GA

In [20]:
def run_real_ga(func, # objective function
                bounds, # list of tuples, 1 per var
                n_gens, # max number of gens to run
                pop_size=100, # number of individuals in the pop
                pc=0.9, # cross prob
                nc=20, # sbx distribution
                nm=20, # poly mutation
                use_elitism=True, # keep best individual across gens
                verbose=False): # print best fitness each gen

  n_vars=len(bounds)
  pm=1.0/n_vars
  stopper=StagnationStopper(patience=50,tol=1e-8)

  #initialization
  pop=init_real_population(pop_size,bounds)
  fitness=evaluate_real_population(pop,func)
  history=[np.min(fitness)]

  #gen loop
  for gen in range(1,n_gens+1):

    #elitism
    if use_elitism:
      elite_idx=np.argmin(fitness)
      elite_ind=pop[elite_idx].copy()
      elite_fit=fitness[elite_idx]

    new_pop=[]
    while len(new_pop)<pop_size:
      # binary tournament selection
      idx1=binary_tournament_selection(fitness)
      idx2=binary_tournament_selection(fitness)
      p1,p2=pop[idx1],pop[idx2]

      # sbx cross
      c1,c2=sbx_crossover(p1,p2,bounds,pc=pc,nc=nc)

      c1=polynomial_mutation(c1,bounds,pm,nm=nm)
      c2=polynomial_mutation(c2,bounds,pm,nm=nm)

      new_pop.append(c1)
      if len(new_pop)<pop_size:
        new_pop.append(c2)

    pop=np.array(new_pop)
    fitness=evaluate_real_population(pop,func)

    # replace worst
    if use_elitism:
      worst_idx=np.argmax(fitness)
      if fitness[worst_idx]>elite_fit:
        pop[worst_idx]=elite_ind
        fitness[worst_idx]=elite_fit

    # record and check
    best_gen=np.min(fitness)
    history.append(best_gen)

    if verbose:
      print(f'gen {gen:4d}|best fitness: {best_gen:.8f}')

    if stopper.should_stop(best_gen):
      if verbose:
        print(f'stopping at gen {gen} (stagnation detected)')
      break

  # return results
  best_idx=np.argmin(fitness)
  return pop[best_idx],fitness[best_idx],history,gen

## 7. Convergence analysis

one run per problem before all 20 experiments to see where the algorithm converges


**n_gens** for all 20 experiments

In [27]:

MAX_GENS = 1000   # max allowed
POP_SIZE  = 100

pilot_results = {}   # to store for each combo

for pname, pconfig in PROBLEMS.items():
    print(f'\n=== {pname} ===')

    # binary
    _, _, hist_bin, gens_bin = run_binary_ga(
        func=pconfig['func'], bounds=pconfig['bounds'],
        n_gens=MAX_GENS, pop_size=POP_SIZE
    )

    # real
    _, _, hist_real, gens_real = run_real_ga(
        func=pconfig['func'], bounds=pconfig['bounds'],
        n_gens=MAX_GENS, pop_size=POP_SIZE
    )

    pilot_results[pname] = {
        'binary': (hist_bin, gens_bin),
        'real'  : (hist_real, gens_real)
    }
    print(f'  Binary stopped at gen {gens_bin} | best = {hist_bin[-1]:.6f}')
    print(f'  Real   stopped at gen {gens_real} | best = {hist_real[-1]:.6f}')


=== TestProblem1 ===


TypeError: 'NoneType' object is not subscriptable